# LIBRARY IMPORTS

In [ ]:
###### LIBRARY IMPORTS ######
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torchaudio
import json
import torch.nn as nn
import time
import matplotlib.pyplot as plt
import re
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Union
from SSM_model_V1 import SSMPhonemeModel as SSMPhonemeModelV1
from SSM_model_V2 import SSMPhonemeModel as SSMPhonemeModelV2
from SSM_model_V3 import SSMPhonemeModel as SSMPhonemeModelV3
from SSM_model_V4 import SSMPhonemeModel as SSMPhonemeModelV4
from SSM_model_V5 import SSMPhonemeModel as SSMPhonemeModelV5
from S2T_CNN_time import S2T_CNN_time_V1
from TIMITDataset import TIMITDataset
import random
from utils import (collate_fn_ctc, save_training_result, collate_fn_ctc_time_domain_padding, 
                   collate_fn_ctc_time_domain_features_during_training, compute_batch_per_from_log_probs, decode_batch_to_phonemes, log_message, 
                   load_phoneme_intervals, calculate_phoneme_mel_statistics, ctc_greedy_decode_ids, 
                   align_phoneme_sequences, check_timit_boundaries)
from functools import partial # for the new collate function
import yaml
from train_eval_utils import (apply_scheduler, calculate_PER_for_batch, calculate_and_update_epoch_loss, 
                              check_gradients_for_inf, create_checkpoint, evaluate_one_batch, 
                              log_batch_loss_and_PER, log_early_stopping, log_epoch_info, 
                              log_training_time, qualitative_predictions, restore_best_model, 
                              stop_early, train_one_batch, calculate_test_dataset_metrics,
                              append_test_results, save_test_results_to_csv, count_invalid_ctc_samples,
                              log_test_phoneme_predictions, initialize_test_predictions_log, print_cuda_memory)
from Decoder import CTCDecoder
from Upsampler import Upsampler
from SSM_TU.patches.scheduler import CosineAnnealingWarmupRestarts
!pip install torch-audiomentations
from Augments import augments_weak

#import torch_xla
#import torch_xla.core.xla_model as xm

seed = 42 # define the seed for pytorch repeatability

# COLAB DEFINITION

In [ ]:
###### DEFINE PATHS EITHER FOR GOOGLE COLAB OR LOCALLY ######
import sys
import os

colab = True # decides whether colab or local pc used

if colab:
    from google.colab import drive
    from google.colab import runtime

    project_root_dir = os.getcwd() # get the root directory of the project
    data_root_dir = os.path.join(project_root_dir, "TIMIT") # define the path to the TIMIT dataset in colab
    drive.mount('/content/drive', force_remount=True) # mount the main drive directory to colab
    sys.path.append('/content/drive/MyDrive/thesis') # to import the SSM_TU directory we need its parent directory
    from SSM_TU.src.model.sequence_layer import SequenceLayer # import the SSM Layer directory
    vocab_root_dir = os.path.join(project_root_dir, "drive", "MyDrive", "thesis")
    ssm_tu_root_dir = vocab_root_dir
    s2s_edge_root_dir = vocab_root_dir

    # unzip the data
    if not os.path.exists("/content/TIMIT"):  # or your dataset folder
        !unzip -q /content/drive/MyDrive/thesis/TIMIT.zip -d /content/
    else:
        print("TIMIT folder already exists in /content/TIMIT folder")

    base_results_dir = r"/content/drive/MyDrive/thesis/results"

    # check the content of the main directory
    print("TIMIT dataset contents")
    !ls /content/TIMIT
else:
    project_root_dir = os.getcwd() # get the root directory of the project
    vocab_root_dir = project_root_dir
    data_root_dir = project_root_dir
    base_results_dir = os.path.join(project_root_dir, "results")
    ssm_tu_root_dir = vocab_root_dir
    s2s_edge_root_dir = vocab_root_dir

vocabulary_dir = os.path.join(vocab_root_dir, "vocabulary")
train_data_csv = os.path.join(data_root_dir, "data", "train_data.csv")
test_data_csv = os.path.join(data_root_dir, "data", "test_data.csv")
s2s_edge_root_dir = os.path.join(s2s_edge_root_dir, "S2S_model_descriptions")
ssm_tu_root_dir = os.path.join(ssm_tu_root_dir, "SSM_TU")

print(f"Project root directory: {project_root_dir}")
print(f"Data root directory: {data_root_dir}")
print(f"Vocabulary directory: {vocabulary_dir}")
print(f"SSM_TU_root_dir: {ssm_tu_root_dir}")
print(f"S2S_model_descriptions: {s2s_edge_root_dir}")
print(f"train_data.csv: {train_data_csv}")
print(f"test_data.csv: {test_data_csv}")
!ls /content/TIMIT/TIMIT


# CUDA DEVICE SPECIFICATIONS

In [ ]:
####### DEVICE SPECIFICATIONS ######

# GPU info (only works if CUDA runtime is active)
!nvidia-smi

complex_use = True  # MPS cannot handle complex tensors

print("PyTorch version:", torch.__version__)

# --- GPU / MPS checks ---
mps_available = torch.backends.mps.is_available()
cuda_available = torch.cuda.is_available()

print("MPS available:", mps_available)
print("MPS built:", torch.backends.mps.is_built())
print("CUDA available:", cuda_available)
print("CUDA built:", torch.backends.cuda.is_built())

# --- TPU check ---
tpu_available = False

#try:
#    tpu_available = True
#    tpu_device = xm.xla_device()
#    print("TPU detected:", tpu_device)

#except ImportError:
#    print("torch_xla not available → TPU not available")


# --- DEVICE SELECTION ---
if cuda_available:
    print("CUDA MEMORY")
    print("-----------")

    print(torch.cuda.get_device_name(0))
    print(f"Total memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    device = torch.device("cuda")
    print("🚀 Using GPU (CUDA)")
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print("Setting seed for cuda")

    torch.backends.cudnn.deterministic = True
    print("Cuda is deterministic")
    torch.backends.cudnn.benchmark = False
    print("Cuda benchmark = Fakse -> uses fixed algorithm") 
    torch.use_deterministic_algorithms(False) # if True, ctc doesnt work
#elif tpu_available:
#    device = tpu_device
#    print("🚀 Using TPU")
elif mps_available and not complex_use:
    device = torch.device("mps")
    print("🚀 Using GPU (MPS)")

else:
    device = torch.device("cpu")
    print("⚠️ Using CPU")

print("Final device:", device)

In [ ]:
####### GPU ALLOCATION DEBUGGING FUNCTIONS #######
import torch


def tensor_memory_mib(tensor: torch.Tensor) -> float:
    """Memory occupied by the tensor's elements in MiB."""
    return tensor.numel() * tensor.element_size() / (1024 ** 2)


def print_dataset_memory(dataset, name: str) -> None:
    """Inspect the waveform tensor stored in a TIMITDataset."""

    print(f"\n===== {name} DATASET =====")

    waveforms = dataset.waveforms

    print(f"Number of recordings: {len(dataset)}")
    print(f"Waveform shape: {tuple(waveforms.shape)}")
    print(f"Waveform dtype: {waveforms.dtype}")
    print(f"Waveform device: {waveforms.device}")

    print(
        f"Waveform tensor memory: "
        f"{tensor_memory_mib(waveforms):.2f} MiB"
    )

    lengths = dataset.orig_waveform_lengths

    total_valid_samples = int(lengths.sum().item())
    total_stored_samples = waveforms.numel()

    padding_samples = (
        total_stored_samples - total_valid_samples
    )

    print(f"Valid audio samples: {total_valid_samples:,}")
    print(f"Stored samples: {total_stored_samples:,}")
    print(f"Padding samples: {padding_samples:,}")

    print(
        f"Padding percentage: "
        f"{100 * padding_samples / total_stored_samples:.2f}%"
    )

    print(
        f"Useful audio memory: "
        f"{total_valid_samples * waveforms.element_size() / 1024**2:.2f} MiB"
    )

    print(
        f"Padding memory: "
        f"{padding_samples * waveforms.element_size() / 1024**2:.2f} MiB"
    )


def print_cuda_memory(stage: str) -> None:
    """Print CUDA memory usage and GPU capacity."""

    print(f"\n===== CUDA MEMORY: {stage} =====")

    if not torch.cuda.is_available():
        print("CUDA is not available.")
        return

    device_index = torch.cuda.current_device()

    torch.cuda.synchronize(device_index)

    properties = torch.cuda.get_device_properties(device_index)

    free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)

    allocated = torch.cuda.memory_allocated(device_index)
    reserved = torch.cuda.memory_reserved(device_index)
    peak = torch.cuda.max_memory_allocated(device_index)

    mib = 1024 ** 2

    print(f"GPU: {properties.name}")
    print(f"Total GPU memory: {total_bytes / mib:.2f} MiB")
    print(f"Free GPU memory: {free_bytes / mib:.2f} MiB")
    print(f"Used GPU memory (all processes): {(total_bytes-free_bytes)/mib:.2f} MiB")

    print(f"PyTorch allocated: {allocated / mib:.2f} MiB")
    print(f"PyTorch reserved: {reserved / mib:.2f} MiB")
    print(f"PyTorch peak allocated: {peak / mib:.2f} MiB")

# VOCABULARY LOADING

In [ ]:
####### LOAD VOCABULARY #######

# Select phoneme vocabulary version.
# Allowed values: 39 or 61
phoneme_set_version = 61

# ---------------------------------------------------------
# Validate phoneme set
# ---------------------------------------------------------
if phoneme_set_version not in {39, 61}:
    raise ValueError(
        f"Unsupported phoneme_set_version={phoneme_set_version}. "
        "Expected either 39 or 61."
    )

# ---------------------------------------------------------
# Define vocabulary filenames
# ---------------------------------------------------------
phoneme_to_idx_vocab_filename = (
    f"phoneme_to_idx_{phoneme_set_version}.json"
)

idx_to_phoneme_vocab_filename = (
    f"idx_to_phoneme_{phoneme_set_version}.json"
)

phoneme_to_idx_vocab_path = os.path.join(
    vocabulary_dir,
    phoneme_to_idx_vocab_filename
)


idx_to_phoneme_vocab_path = os.path.join(
    vocabulary_dir,
    idx_to_phoneme_vocab_filename
)

# ---------------------------------------------------------
# Check if the vocabulary files exist
# ---------------------------------------------------------
if not os.path.exists(phoneme_to_idx_vocab_path):
    raise FileNotFoundError(
        f"Vocabulary file not found:\n{phoneme_to_idx_vocab_path}"
    )

if not os.path.exists(idx_to_phoneme_vocab_path):
    raise FileNotFoundError(
        f"Vocabulary file not found:\n{idx_to_phoneme_vocab_path}"
    )

# ---------------------------------------------------------
# Load phoneme -> index vocabulary
# ---------------------------------------------------------
with open(phoneme_to_idx_vocab_path, "r", encoding="utf-8") as file:
    phoneme_to_idx_vocab = json.load(file)

# ---------------------------------------------------------
# Load index -> phoneme vocabulary
# ---------------------------------------------------------
with open(idx_to_phoneme_vocab_path, "r", encoding="utf-8") as file:
    idx_to_phoneme_vocab = json.load(file)

# JSON stores dictionary keys as strings, therefore convert them back to integers.
idx_to_phoneme_vocab = {
    int(idx): phoneme
    for idx, phoneme in idx_to_phoneme_vocab.items()
}

print(
    f"Successfully loaded the {phoneme_set_version}-phoneme vocabulary.\n"
    f"Vocabulary size (including CTC blank): {len(phoneme_to_idx_vocab)}"
)

# DEFINE MODEL/LOSS/OPTIMIZER

In [ ]:
####### model parameters checking function #######
def check_model_parameters(model):

    for name, param in model.named_parameters():

        if not torch.isfinite(param).all():

            print(
                f"NON-FINITE PARAMETER: {name} | "
                f"NaNs={torch.isnan(param).sum().item()} | "
                f"Infs={torch.isinf(param).sum().item()}"
            )

            raise RuntimeError(
                f"Model parameter became non-finite: {name}"
            )

In [ ]:
####### Training relevant objects ######
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

use_mel_transform = False

#The Mel spectrogram does the following:
#1. Split signal into windows
#2. Apply Fourier Transform (FFT)
#3. Convert frequencies to Mel scale
#4. Output energy per Mel band
mel_transform_n_mels = 80 # number of features in each timing window after transformation
mel_transform_nfft = 300
mel_transform_hop_length = 160
mel_transform_win_length = 300
mel_transform_sample_rate = 16000

epochs = 1
model_dict = {
    "model_version": "SSMPhonemeModelV4",
}
ssm_vocab_size = len(phoneme_to_idx_vocab)
ssm_dropout = 0.0
linear_dropout = 0.2 # best -> 0.2

step_scale_config_name = "Custom_subsampling"

if use_mel_transform:
    d_in_first_layer = mel_transform_n_mels
else:
    d_in_first_layer = 1
    

yaml_name = "S2S_Edge_small.yaml"
yaml_path = os.path.join(s2s_edge_root_dir, yaml_name)

with open(yaml_path, "r") as file:
    model_config = yaml.safe_load(file)
    
    d_out = model_config["output_sizes"].copy()
    d_state = model_config["hidden_sizes"].copy()  
    
    trainable_SkipLayer = model_config["trainable_skip_connections"]
    
    # if trainable_Skiplayer is false then d_in and d_out must have to look the same -> mel_transform_n_mels must be changed
    if not trainable_SkipLayer and use_mel_transform:
        mel_transform_n_mels = d_out[0]
        d_in_first_layer = mel_transform_n_mels

    d_in = [d_in_first_layer, *d_out[:-1]]

    input_bias = model_config["input_bias"]
    output_bias = model_config["output_bias"]
    complex_output = model_config["complex_output"]
    act = model_config["act"]
    norm = model_config["norm"]
    norm_type = model_config["norm_type"]
    stability = model_config["stability"]
    
    if step_scale_config_name is not None:
        try:
            step_scales = model_config["step_scale_configurations"][step_scale_config_name]
        except KeyError as k:
            step_scales = [1,2,256]
    else:
        step_scales = [1] * len(d_out)

    rzoh_probabilities = model_config["zeroOrderHoldRegularization"]
    if len(rzoh_probabilities) == 0: # rzoh prob values list shouldnt be empty
        rzoh_probabilities = None # None is handled inside the class definition

    B_C_init = model_config["B_C_init"]
    bias_init = model_config["bias_init"]
    ssm_num_layers = len(d_out)

########## MODEL DEFINITION ##########
print_cuda_memory("A. Before model creation")
ssm_model = SSMPhonemeModelV4(d_state=d_state,
                              d_in=d_in,
                              d_out=d_out,
                              num_of_ssm_layers=ssm_num_layers,
                              vocab_size=ssm_vocab_size,
                              ssm_dropout=ssm_dropout,
                              input_bias = input_bias,
                              output_bias = output_bias,
                              complex_output = complex_output,
                              B_C_init=B_C_init,
                              bias_init = bias_init,
                              stability=stability,
                              linear_dropout=linear_dropout,
                              norm=norm,
                              norm_type=norm_type,
                              act=act,
                              trainable_SkipLayer=trainable_SkipLayer,
                              rzoh_probabilities=rzoh_probabilities)
ssm_model.set_step_scale(step_scales=step_scales)
print_cuda_memory("B. After model creation")
ssm_model.to(device=device)
print_cuda_memory("C. After moving model to CUDA")

# transformation definition is later because of trainable_SkipLayer and d_in_first_layer definition
########## MEL TRANSFORMATION and DICTIOMARY DEFINITION ##########
if use_mel_transform:
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=mel_transform_sample_rate,
        n_fft=mel_transform_nfft, # FFT window size => how many samples are used for frequency analysis
        hop_length=mel_transform_hop_length, # step size between windows => at 16kHz a hop length of 160 means a stepsize of 10ms
        win_length=mel_transform_win_length, # Actual window size applied before FFT => at 16kHz a window size of 400 means 25ms window length
        n_mels=mel_transform_n_mels # Number of frequency bins after Mel conversion => number of frequency elements in each window AFTER conversion
    )
    # mel spectrogram settings
    mel_transform_dict = {
        "mel_transform_n_mels": mel_transform_n_mels,
        "mel_transform_nfft": mel_transform_nfft,
        "mel_transform_hop_length": mel_transform_hop_length,
        "mel_transform_win_length": mel_transform_win_length,
        "mel_transform_sample_rate": mel_transform_sample_rate,
    }
    print(f"✅ Mel spectrogram transformation is used with the following settings: {mel_transform_dict}")
else: 
    mel_transform_dict = None
    print(f"✅ Mel spectrogram transformation is  NOT used")
##################################################################

apply_rzoh = True if rzoh_probabilities is not None else False
model_dict.update({
    "ssm_d_in": d_in,
    "ssm_d_state": d_state,
    "ssm_d_out": d_out,
    "num_of_ssm_layers": ssm_num_layers,
    "linear_dropout": linear_dropout,
    "norm": norm,
    "ssm_dropout": ssm_dropout,
    "input_bias": input_bias,
    "output_bias": output_bias,
    "complex_output": complex_output,
    "B_C_init": B_C_init,
    "bias_init": bias_init,
    "stability": stability,
    "norm_type": norm_type,
    "act": act,
    "trainable_SkipLayer": trainable_SkipLayer,
    "apply_rzoh": apply_rzoh,
    "rzoh_probabilities": rzoh_probabilities,
    "sub_sampling_config_name": step_scale_config_name,
    "sub_sampling": step_scales.copy()
})
################################################
print(f"✅ Input dimension: {d_in}")
print(f"✅ Output dimension: {d_out}")

# CTC loss function
ctc_zero_infinity = False
loss_fn = nn.CTCLoss(blank=0,
                     zero_infinity=ctc_zero_infinity) # to nullyify the infinity effects of possible very short sequences

# optimizer
optimizer_type = "AdamW"
optimizer_dict = {}
if optimizer_type == "Adam":
    opt_Adam_lr = 0.001
    opt_Adam_weight_decay = 1e-4
    optimizer = torch.optim.Adam(
        params=ssm_model.parameters(),
        lr = opt_Adam_lr,
        weight_decay=opt_Adam_weight_decay)
    
    
    optimizer_dict = {
        "type": optimizer_type,
        "learning_rate": opt_Adam_lr,
        "weight_decay": opt_Adam_weight_decay,
    }

elif optimizer_type == "AdamW":
    opt_AdamW_other_layer_weight_decay = 0.01
    opt_AdamW_ssm_layer_weight_decay = 0.0
    
    opt_AdamW_lr = 0.0005
    opt_AdamW_ssm_lr_factor = 1
    opt_AdamW_other_lr_factor = 4

    params_ssm_lr = [param for name, param in ssm_model.named_parameters() if 'B' in name or 'C' in name or 'Lambda' in name or 'log_step' in name] # special SSM parameters B, C, Lambda and log_step
    params_other_lr = [param for name, param in ssm_model.named_parameters() if 'B' not in name and 'C' not in name and 'Lambda' not in name and 'log_step' not in name] # parameters of other layers like the Linear Decoder

    optimizer_dict = {
        "type": optimizer_type,
        "learning_rate": opt_AdamW_lr,
        "ssm_lr_factor": opt_AdamW_ssm_lr_factor,
        "other_lr_factor": opt_AdamW_other_lr_factor,
        "ssm_weight_decay": opt_AdamW_ssm_layer_weight_decay,
        "other_weight_decay": opt_AdamW_other_layer_weight_decay,
    }


    optimizer = torch.optim.AdamW([
            {'params': params_ssm_lr, 'lr':opt_AdamW_ssm_lr_factor * opt_AdamW_lr, 'weight_decay': opt_AdamW_ssm_layer_weight_decay},
            {'params': params_other_lr, 'lr': opt_AdamW_other_lr_factor * opt_AdamW_lr, 'weight_decay': opt_AdamW_other_layer_weight_decay},
        ], 
    lr=opt_AdamW_lr, weight_decay=opt_AdamW_other_layer_weight_decay) # default values for lr and weight decay

print(f"✅ SSMPhonemeModelV4 model is used")
print_cuda_memory("D. After optimizer creation")

# Scheduler
scheduler_type = "CosineAnnealingWarmUp" # StepLR, ReduceLROnPlateau,
scheduler_dict = {}                     

# StepLR parameters
StepLR_scheduler_step_size = 25
StepLR_scheduler_gamma = 0.5

# ReduceLROnPlateau parameters
plateau_scheduler_mode = "min"
plateau_scheduler_factor = 0.3
plateau_scheduler_patience = 3
plateau_scheduler_threshold = 1e-3
plateau_scheduler_minlr = 1e-5



if scheduler_type == "StepLR":
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer,
                                                step_size=StepLR_scheduler_step_size,
                                                gamma=StepLR_scheduler_gamma)
    scheduler_dict = {
        "scheduler_type": scheduler_type,
        "step_size": StepLR_scheduler_step_size,
        "gamma": StepLR_scheduler_gamma
    }
    print("✅ StepLR scheduler is used")

elif scheduler_type == "ReduceLROnPlateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode=plateau_scheduler_mode,
        factor=plateau_scheduler_factor,
        patience=plateau_scheduler_patience,
        threshold=plateau_scheduler_threshold,
        min_lr=plateau_scheduler_minlr
    )
    scheduler_dict = {
            "scheduler_type": scheduler_type,
            "mode": plateau_scheduler_mode,
            "factor": plateau_scheduler_factor,
            "patience": plateau_scheduler_patience,
            "threshold": plateau_scheduler_threshold,
            "min_lr": plateau_scheduler_minlr
        }
    print("✅ ReduceLROnPlateu scheduler is used")


# Gradient clipping
use_gradient_clipping = True
print("step_scales used:", step_scales)


# DATASETS/DATALOADERS

In [ ]:
####### DATASETS AND DATALOADERS #######

print_cuda_memory("BEFORE DATASET CREATION")

padding_after_mel_transform = False

collate_function = ""

if padding_after_mel_transform:

    dataloader_collate_fn = collate_fn_ctc
    collate_function = "collate_fn_ctc"

else:

    if not use_mel_transform:
        dataloader_collate_fn = (
            collate_fn_ctc_time_domain_features_during_training
        )

        collate_function = (
            "collate_fn_ctc_time_domain_features_during_training"
        )
        mel_transform_dict = None
        print(f"✅ Mel spectrogram transformation is NOT used")
    else:
        dataloader_collate_fn = partial(
            collate_fn_ctc_time_domain_padding,
            mel_transform=mel_transform,
            hop_length=mel_transform_hop_length,
            apply_rzoh=apply_rzoh
        )

        collate_function = "collate_fn_ctc_time_domain_padding"
        print(f"✅ Mel spectrogram transformation is used")


# ============================================================
# TRAINING DATASET
# ============================================================

batch_size = 8

train_dataset = TIMITDataset(
    csv_file=train_data_csv,
    train_or_test="TRAIN",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=None,
)

print_dataset_memory(train_dataset, "TRAIN")
print_cuda_memory("AFTER TRAIN DATASET")


# ============================================================
# VALIDATION DATASET
# ============================================================

val_dataset = TIMITDataset(
    csv_file=test_data_csv,
    train_or_test="TEST",
    root_dir=project_root_dir,
    phoneme_to_idx_vocab=phoneme_to_idx_vocab,
    transform=None,
)

print_dataset_memory(val_dataset, "VALIDATION")
print_cuda_memory("AFTER VALIDATION DATASET")


# ============================================================
# DATALOADERS
# ============================================================

g = torch.Generator()
g.manual_seed(seed)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=dataloader_collate_fn,
    generator=g
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=dataloader_collate_fn
)

# if the scheduler is CosineAnnealingWarmup
# it needs the length of the train dataloader
# CosineAnnealingWarmUp parameters
cosanneal_first_cycle_steps = epochs * len(train_dataloader)
cosanneal_cycle_mult = 1.0
cosanneal_max_lr = [opt_AdamW_ssm_lr_factor * opt_AdamW_lr,
                    opt_AdamW_other_lr_factor * opt_AdamW_lr]
cosanneal_min_lr = [0, 0]
cosanneal_warmup_steps = 100
cosanneal_gamma = 1

if scheduler_type == "CosineAnnealingWarmUp":
    scheduler = CosineAnnealingWarmupRestarts(
        optimizer,
        first_cycle_steps=cosanneal_first_cycle_steps,
        cycle_mult=cosanneal_cycle_mult,
        max_lr=cosanneal_max_lr,
        min_lr=cosanneal_min_lr,
        warmup_steps=cosanneal_warmup_steps,
        gamma=cosanneal_gamma,
    )

    scheduler_dict = {
        "scheduler_type": scheduler_type,
        "first_cycle_steps": cosanneal_first_cycle_steps,
        "cycle_mult":cosanneal_cycle_mult,
        "max_lr":cosanneal_max_lr,
        "min_lr":cosanneal_min_lr,
        "warmup_steps":cosanneal_warmup_steps,
        "gamma":cosanneal_gamma,
    }
    print("✅ CosineAnnealingWarmuUpRestart scheduler is used")

print_cuda_memory("AFTER DATALOADER CREATION")


# DECODER

In [ ]:
####### DEFINE THE DECODER #######
blank_idx = 0
beam_width = 5
top_paths = 1
decoding_method = "beam_search" # Options: 1. greedy
                           #          2. beam_search
ctc_decoder = CTCDecoder(blank_idx=blank_idx,
                         beam_width=beam_width,
                         top_paths=top_paths,
                         method=decoding_method)

print(f"✅ Decoder beam width: {beam_width}")
print(f"✅ Decoder top paths: {top_paths}")
print(f"✅ Decoder decoding method: {decoding_method}")


# TRAINING

In [ ]:
######### Checkpoint definition function #########
def load_model_from_checkpoint(use_scheduler: bool) -> tuple:

    checkpoint = torch.load(
        "checkpoint.pt",
        map_location=device,
        weights_only=True
    )

    ssm_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    if use_scheduler:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    start_epoch = checkpoint["epoch"] + 1

    train_epoch_losses = checkpoint.get("train_epoch_losses", [])
    val_epoch_losses = checkpoint.get("val_epoch_losses", [])
    train_epoch_PERs = checkpoint.get("train_epoch_PERs", [])
    val_epoch_PERs = checkpoint.get("val_epoch_PERs", [])

    if all(
        key in checkpoint
        for key in (
            "best_val_loss",
            "best_epoch",
            "epochs_without_improvement"
        )
    ):
        # New checkpoint format
        best_val_loss = checkpoint["best_val_loss"]
        best_epoch = checkpoint["best_epoch"]
        epochs_without_improvement = checkpoint["epochs_without_improvement"]

    else:
        # Old checkpoint format: reconstruct early-stopping state
        best_val_loss = float("inf")
        best_epoch = -1
        epochs_without_improvement = 0

        for epoch_idx, loss in enumerate(val_epoch_losses):

            improvement = best_val_loss - loss

            if improvement > early_stopping_min_delta:
                best_val_loss = loss
                best_epoch = epoch_idx
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

    return (
        start_epoch,
        train_epoch_losses,
        val_epoch_losses,
        train_epoch_PERs,
        val_epoch_PERs,
        best_val_loss,
        best_epoch,
        epochs_without_improvement
    )

In [ ]:
######### Training and Validation loop with optional PER #########
training_start_time = time.time()

use_scheduler = True
compute_per = True  # set to False to skip PER computation and PER logging for saving time
resume_from_checkpoint = False

epoch_per_calc_division = 1

use_early_stopping = True

early_stopping_patience = 8
early_stopping_min_delta = 1e-4


best_val_loss = float("inf")
best_epoch = -1
epochs_without_improvement = 0

best_model_path = "best_model_state_dict.pt"

print("device: ", device)
print("batch_size: ",batch_size)
print("number of ssm layers:", ssm_num_layers)
print("Scheduler: ", scheduler_type)
print("Use Checkpoint: ", resume_from_checkpoint)

blank_idx = 0   # CTC blank index

if resume_from_checkpoint:
    (start_epoch,
     train_epoch_losses,
     val_epoch_losses,
     train_epoch_PERs,
     val_epoch_PERs
    ) = load_model_from_checkpoint(use_scheduler=use_scheduler)
    
    file_mode = "a"
else:
    start_epoch = 0
    train_epoch_losses = []
    val_epoch_losses = []
    train_epoch_PERs = []
    val_epoch_PERs = []

    file_mode="w"

log_file_loss_and_PER_path = "training_log_loss_and_PER.txt"
log_file_predictions_path = "training_log_predictions.txt"
log_file_loss_and_PER = open(log_file_loss_and_PER_path, file_mode)
log_message(
    f"scheduler configuration: {scheduler_dict}",
    log_file_loss_and_PER
)
log_file_predictions = open(log_file_predictions_path, file_mode)

# DEBUG CUDA
debug_cuda_memory = False
debug_batches = 3
# DEBUG CUDA


for epoch in range(start_epoch, epochs):
    ######### Reinitalize batch and epoch relevant quantities ###############
    true_train_batch_loss = 0.0
    epoch_train_loss = 0.0

    true_val_batch_loss = 0.0
    epoch_val_loss = 0.0

    total_train_edit_distance = 0
    total_train_target_length = 0
    epoch_train_per = None

    total_val_edit_distance = 0
    total_val_target_length = 0
    epoch_val_per = None


    ###################################################################################################################################
    ############################################################# TRAINING ############################################################
    ssm_model.train()

    for batch, (X_train, X_train_length, y_train, y_train_length) in enumerate(train_dataloader):
        ########## Trains one batch ##########
        batch_train_per = None # necessary to define for logging

        # DEBUG CUDA
        debug_this_batch = (
            debug_cuda_memory
            and epoch == start_epoch
            and batch < debug_batches
        )

        log_probs, X_train_length, log_probs_ctc, true_train_batch_loss, train_loss = train_one_batch(X_train=X_train,
                                                                                      X_train_length=X_train_length,
                                                                                      y_train=y_train,
                                                                                      y_train_length=y_train_length,
                                                                                      device=device,
                                                                                      ssm_model=ssm_model,
                                                                                      loss_fn=loss_fn,
                                                                                      true_train_batch_loss=true_train_batch_loss,
                                                                                      debug_memory=debug_this_batch,
                                                                                      augmentation=augments_weak)

        # catch learning rates for experiments
        batch_ssm_lr = optimizer.param_groups[0]["lr"]
        batch_other_lr = optimizer.param_groups[1]["lr"]
        # DEBUG CUDA
        
        
        # log_probs, log_probs_ctc, true_train_batch_loss, train_loss = train_one_batch(X_train=X_train,
        #                                                                               X_train_length=X_train_length,
        #                                                                               y_train=y_train,
        #                                                                               y_train_length=y_train_length,
        #                                                                               device=device,
        #                                                                               ssm_model=ssm_model,
        #                                                                               loss_fn=loss_fn,
        #                                                                               true_train_batch_loss=true_train_batch_loss)

        ########## compute PER for the current training batch -> IF ENABLED ##########
        if compute_per and (epoch == 0 or (epoch + 1) % epoch_per_calc_division == 0):
            batch_train_per, total_train_edit_distance, total_train_target_length = calculate_PER_for_batch(log_probs=log_probs,
                                                                                                            X_length=X_train_length,
                                                                                                            y=y_train,
                                                                                                            y_length=y_train_length,
                                                                                                            total_edit_distance=total_train_edit_distance,
                                                                                                            total_target_length=total_train_target_length,
                                                                                                            decoder=ctc_decoder)

        # DEBUG CUDA
        optimizer.zero_grad(set_to_none=True)

        if debug_this_batch:
            print_cuda_memory("9. Before backward")

        train_loss.backward()

        if debug_this_batch:
            print_cuda_memory("10. After backward")
        # DEBUG CUDA
        
        # ########## backpropagation ##########
        # optimizer.zero_grad()
        # train_loss.backward()

        ########## Check gradients BEFORE clipping ##########
        check_gradients_for_inf(ssm_model=ssm_model)

        ########## Gradient clipping ##########
        if use_gradient_clipping:
            grad_norm = torch.nn.utils.clip_grad_norm_(
                ssm_model.parameters(),
                max_norm=1.0
            )

            if not torch.isfinite(grad_norm):
                raise RuntimeError(
                    f"NON-FINITE GRADIENT NORM: {grad_norm}"
                )

        ########## move the optimizer (gradient descent) ##########
        # DEBUG CUDA
        if debug_this_batch:
            print_cuda_memory("11. Before optimizer.step")

        optimizer.step()

        # CosineAnneal scheduler is called after every batch
        if use_scheduler and scheduler_type == "CosineAnnealingWarmUp":
            scheduler.step()

        if debug_this_batch:
            print_cuda_memory("12. After optimizer.step")
        # DEBUG CUDA
        
        #optimizer.step()
        check_model_parameters(ssm_model)

        ########## log the training batch loss and batch PER (if enabled) into the LOG file ##########
        if batch % 30 == 0: 
            log_batch_loss_and_PER(batch=batch,
                                   compute_per=compute_per,
                                   epoch=epoch,
                                   epoch_per_calc_division=epoch_per_calc_division,
                                   dataloader_obj=train_dataloader,
                                   loss=train_loss,
                                   batch_per=batch_train_per,
                                   log_file_loss_and_PER=log_file_loss_and_PER,
                                   phase="Training",
                                   learning_rates = (batch_ssm_lr, batch_other_lr),
                                   cosanneal_warmup_steps = cosanneal_warmup_steps)

    ################## CALCULATE VALIDATION LOSS FOR WHOLE EPOCH ##################
    epoch_train_loss = calculate_and_update_epoch_loss(true_batch_loss=true_train_batch_loss,
                                                       dataset=train_dataset,
                                                       epoch_losses_list=train_epoch_losses)

    ################## CALCULATE THE TRAINING PER VALUE FOR THE WHOLE EPOCH IF ENABLED ##################
    if compute_per and (epoch == 0 or (epoch + 1) % epoch_per_calc_division == 0):
        epoch_train_per = (
            total_train_edit_distance / total_train_target_length
            if total_train_target_length > 0 else 0.0
        )
        train_epoch_PERs.append(epoch_train_per)
    else:
        train_epoch_PERs.append(None)

    
    ############################################################ VALIDATION ######################################################################
    ##############################################################################################################################################
    ssm_model.eval()

    with torch.inference_mode():
        for batch, (X_val, X_val_length, y_val, y_val_length) in enumerate(val_dataloader):
            ########## evaluate one validation batch ##########
            batch_val_per=None # necessary to define for logging
            log_probs, log_probs_ctc, valid_output_lengths, true_val_batch_loss, val_loss  = evaluate_one_batch(X_val=X_val,
                                                                                                                X_val_length=X_val_length,
                                                                                                                y_val=y_val,
                                                                                                                y_val_length=y_val_length,
                                                                                                                device=device,
                                                                                                                ssm_model=ssm_model,
                                                                                                                loss_fn=loss_fn,
                                                                                                                true_val_batch_loss=true_val_batch_loss)

            ########## compute validation PER for this batch - if enabled ##########
            if compute_per and (epoch == 0 or (epoch + 1) % epoch_per_calc_division == 0):
                batch_val_per, total_val_edit_distance, total_val_target_length = calculate_PER_for_batch(log_probs=log_probs,
                                                                                                          X_length=valid_output_lengths,
                                                                                                          y=y_val,
                                                                                                          y_length=y_val_length,
                                                                                                          total_edit_distance=total_val_edit_distance,
                                                                                                          total_target_length=total_val_target_length,
                                                                                                          decoder=ctc_decoder)

            ########## log the batch loss and batch PER (if necessary) into the LOG file ##########
            if batch % 10 == 0:
                log_batch_loss_and_PER(batch=batch,
                                       compute_per=compute_per,
                                       epoch=epoch,
                                       epoch_per_calc_division=epoch_per_calc_division,
                                       dataloader_obj=val_dataloader,
                                       loss=val_loss,
                                       batch_per=batch_val_per,
                                       log_file_loss_and_PER=log_file_loss_and_PER,
                                       phase="Validation",
                                       learning_rates = (batch_ssm_lr, batch_other_lr),
                                       cosanneal_warmup_steps=cosanneal_warmup_steps)

    ################## CALCULATE VALIDATION LOSS FOR WHOLE EPOCH ##################
    epoch_val_loss = calculate_and_update_epoch_loss(true_batch_loss=true_val_batch_loss,
                                                     dataset=val_dataset,
                                                     epoch_losses_list=val_epoch_losses)
    
    ################## CALCULATE THE VALIDATION PER VALUE FOR THE WHOLE EPOCH IF ENABLED ##################
    if compute_per and (epoch == 0 or (epoch + 1) % epoch_per_calc_division == 0):
        epoch_val_per = (
            total_val_edit_distance / total_val_target_length
            if total_val_target_length > 0 else 0.0
        )
        val_epoch_PERs.append(epoch_val_per)
    else:
        val_epoch_PERs.append(None)

    
    ################## EARLY STOPPING ##################
    should_stop_early = False
    best_val_loss, best_epoch, epochs_without_improvement, should_stop_early = stop_early(use_early_stopping=use_early_stopping,
                                                                                          best_epoch=best_epoch,
                                                                                          best_val_loss=best_val_loss,
                                                                                          epoch_val_loss=epoch_val_loss,
                                                                                          early_stopping_min_delta=early_stopping_min_delta,
                                                                                          epoch=epoch,
                                                                                          epochs_without_improvement=epochs_without_improvement,
                                                                                          early_stopping_patience=early_stopping_patience,
                                                                                          ssm_model=ssm_model,
                                                                                          best_model_path=best_model_path,
                                                                                          log_file_loss_and_PER=log_file_loss_and_PER)

    ################## USE SCHEDULER IF NECESSARY ##################
    if use_scheduler and scheduler_type != "CosineAnnealingWarmUp":
        apply_scheduler(scheduler_type=scheduler_type,
                        scheduler=scheduler,
                        epoch_val_loss=epoch_val_loss,
                        optimizer=optimizer)

    ################## CHECKPOINT CREATION AFTER EVERY EPOCH ##################
    create_checkpoint(epoch=epoch,
                      use_scheduler=use_scheduler,
                      train_epoch_losses=train_epoch_losses,
                      val_epoch_losses=val_epoch_losses,
                      train_epoch_PERs=train_epoch_PERs,
                      val_epoch_PERs=val_epoch_PERs,
                      ssm_model=ssm_model,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      best_val_loss=best_val_loss,
                      best_epoch=best_epoch,
                      epochs_without_improvement=epochs_without_improvement)

    ################## SAVE THE EPOCH LOSS AND PER FOR TRAINING AND VALIDATION ##################
    log_epoch_info(compute_per=compute_per,
                   epoch=epoch,
                   epoch_per_calc_division=epoch_per_calc_division,
                   log_file_loss_and_PER=log_file_loss_and_PER,
                   epoch_train_loss=epoch_train_loss,
                   epoch_train_per=epoch_train_per,
                   epoch_val_loss=epoch_val_loss,
                   epoch_val_per=epoch_val_per)

    ################## QUALITATIVE VALIDATION PREDICTIONS EVERY 10 EPOCHS ##################
    qualitative_predictions(epoch=epoch,
                            predict_every_epoch=1,
                            ssm_model=ssm_model,
                            log_file_predictions=log_file_predictions,
                            val_dataloader=val_dataloader,
                            idx_to_phoneme_vocab=idx_to_phoneme_vocab,
                            device=device,
                            blank_idx=blank_idx,
                            decoder=ctc_decoder)

    ################## LOG EARLY STOPPING IF NECESSARY ##################
    if should_stop_early:
        log_early_stopping(epoch=epoch,
                           best_epoch=best_epoch,
                           best_val_loss=best_val_loss,
                           log_file_loss_and_PER=log_file_loss_and_PER,
                           log_file_predictions=log_file_predictions)
        break

################## RESTORE BEST MODEL ##################
if use_early_stopping and best_epoch >= 0:
    restore_best_model(best_model_path=best_model_path,
                       device=device,
                       ssm_model=ssm_model,
                       best_epoch=best_epoch,
                       best_val_loss=best_val_loss)

################## log and print the elapsed training time ##################
elapsed_training_time_str = log_training_time(training_start_time=training_start_time,
                                              log_file_loss_and_PER=log_file_loss_and_PER,
                                              log_file_predictions=log_file_predictions)

# close the file handlers of the log files
log_file_loss_and_PER.close()
log_file_predictions.close()
print(f"✅ Training successfully finished")



######################################################## PLOT THE LOSS AND PER VALUES ########################################################
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
axes[0].plot(train_epoch_losses, label="Training loss")
axes[0].plot(val_epoch_losses, label="Validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CTC Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()

# PER plot
if compute_per == True:
    train_epoch_PERs_percent = [per * 100 for per in train_epoch_PERs]
    val_epoch_PERs_percent = [per * 100 for per in val_epoch_PERs]
    axes[1].plot(train_epoch_PERs_percent, label="Training PER")
    axes[1].plot(val_epoch_PERs_percent, label="Validation PER")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("PER (%)")
    axes[1].set_title("Training and Validation PER")
    axes[1].legend()

fig.tight_layout()
plt.show()
print(f"✅ Plots successfully created")
################################################################################################################################################


######################################################## CREATE THE TRAINING CONFIGURATIONS DICTIONARY ###################################################
min_train_epoch_loss = min(train_epoch_losses)
min_val_epoch_loss = min(val_epoch_losses)

valid_train_PERs = [per for per in train_epoch_PERs if per is not None]
valid_val_PERs = [per for per in val_epoch_PERs if per is not None]

min_train_epoch_PER = min(valid_train_PERs) if valid_train_PERs else None
min_val_epoch_PER = min(valid_val_PERs) if valid_val_PERs else None

experiment_config = {
    # training loop
    "epochs": epochs,
    "use_scheduler": use_scheduler,
    "training_time:": elapsed_training_time_str,
    "min_train_epoch_loss": min_train_epoch_loss,
    "min_val_epoch_loss": min_val_epoch_loss,
    "min_train_epoch_PER": min_train_epoch_PER if min_train_epoch_PER is not None else "PER not calculated",
    "min_val_epoch_PER": min_val_epoch_PER if min_val_epoch_PER is not None else "PER not calculated",
    
    "batch_size": batch_size,
    "seed": seed,

    "collate_function": collate_function,

    # Dataset settings
    "padding_after_mel_transform": padding_after_mel_transform,
    "collate_function": collate_function,

    # Mel spectrogram settings
    "mel_transform_dict": mel_transform_dict,

    # model settings
    "apply_rzoh": apply_rzoh,
    "model_info": model_dict,
    
    # loss settings
    "ctc_zero_infinity": ctc_zero_infinity,
    "blank_index": 0,

    # optimizer settings
    "optimizer": optimizer_dict,

    # scheduler settings
    "scheduler_dict": scheduler_dict,

    # gradient_clipping
    "use_gradient_clipping": use_gradient_clipping,

    # vocabulary
    "phoneme_to_idx_vocab": phoneme_to_idx_vocab,
}

losses_and_accuracies_config = {
    "train_epoch_losses": train_epoch_losses,
    "train_epoch_accuracies": train_epoch_PERs,
    "val_epoch_losses": val_epoch_losses,
    "val_epoch_accuracies": val_epoch_PERs,
}
print(f"✅ results dict successfully created")
######################################################################################################################################################

######################################################## SAVE THE RESULTS ########################################################
result_dir = save_training_result(
    figures=fig,
    model=ssm_model,
    config=experiment_config,
    base_results_dir=base_results_dir,
    log_files=[
        log_file_loss_and_PER_path,
        log_file_predictions_path
    ]
)

losses_and_accuracies_path = Path(result_dir) / "losses_and_accuracies.json"
with open(losses_and_accuracies_path, "w") as f:
    json.dump(losses_and_accuracies_config, f, indent=4)

print(f"✅ Saved training results to: {result_dir}")
####################################################################################################################################

####################################################### REASSIGN RUNTIME ###########################################################
runtime.unassign()
print(f"✅ Runtime kernel probably dropped succesfully")
